In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import custom CRUD Python module
# Ensure this matches your file name (e.g., animal_shelter or CRUD_Python_Module)
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Authenticate using your credentials created in Module Three
username = "aacuser"
password = "Prince0000"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Retrieve all initial records
df = pd.DataFrame.from_records(db.read({}))

# Drop MongoDB ObjectID column to prevent JSON serialization errors in DataTable
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load and encode Grazioso Salvare Logo
# Update file path/name if stored in a subfolder like 'code_files/Grazioso Salvare Logo.png'
image_filename = 'Grazioso Salvare Logo.png'
if not os.path.exists(image_filename) and os.path.exists('code_files/Grazioso Salvare Logo.png'):
    image_filename = 'code_files/Grazioso Salvare Logo.png'

encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

app.layout = html.Div([
    # Client Branding and Unique Identifier Header
    html.Div([
        html.A(
            html.Img(
                src=f'data:image/png;base64,{encoded_image}',
                style={'height': '120px', 'width': 'auto', 'display': 'block', 'margin': 'auto'}
            ),
            href='https://www.snhu.edu',
            target='_blank',
            title='Visit SNHU'
        ),
        html.H1('Grazioso Salvare Rescue Dog Identification Dashboard', style={'textAlign': 'center', 'marginTop': '10px'}),
        html.H3('Developed by: Michael Rickerson', style={'textAlign': 'center', 'color': '#555'})
    ], style={'padding': '15px'}),

    html.Hr(),

    # Interactive Radio Button Filter Options
    html.Div([
        html.Label('Select Rescue Filter:', style={'fontWeight': 'bold', 'marginRight': '15px', 'fontSize': '16px'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water Rescue'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain Rescue'},
                {'label': 'Disaster Rescue or Individual Tracking', 'value': 'Disaster Rescue'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'marginRight': '25px', 'cursor': 'pointer'}
        )
    ], style={'padding': '10px 20px'}),

    html.Hr(),

    # Interactive Data Table with Sorting, Filtering, and Pagination
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto', 'padding': '10px'},
        style_cell={'textAlign': 'left', 'padding': '5px', 'minWidth': '100px'},
        style_header={'backgroundColor': '#f2f2f2', 'fontWeight': 'bold'}
    ),

    html.Br(),
    html.Hr(),

    # Visualization Row: Side-by-Side Pie Chart and Geolocation Map
    html.Div(
        className='row',
        style={'display': 'flex', 'flexWrap': 'wrap'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'flex': '1', 'minWidth': '450px', 'padding': '10px'}
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'flex': '1', 'minWidth': '450px', 'padding': '10px'}
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

# Callback to filter data table according to selected rescue criteria
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    query = {}
    
    if filter_type == 'Water Rescue':
        query = {
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_type == 'Mountain Rescue':
        query = {
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_type == 'Disaster Rescue':
        query = {
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20.0, "$lte": 300.0}
        }
    elif filter_type == 'Reset':
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
        
    return dff.to_dict('records')


# Callback to update the Pie Chart reflecting filtered breeds
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [html.P("No records available to display chart.")]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    fig = px.pie(
        dff,
        names='breed',
        title='Preferred Dog Breeds Distribution',
        color_discrete_sequence=px.colors.qualitative.Dark24
    )
    return [dcc.Graph(figure=fig)]


# Callback to highlight a selected column
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'backgroundColor': '#D2F3FF'
    } for i in selected_columns]


# Callback to update the Leaflet Geolocation Map based on the selected table row
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return [html.P("No data available for geolocation map.")]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Default to first row if none selected
    row = 0 if not index else index[0]
    
    # Fallback bounds check if filtered data has fewer rows
    if row >= len(dff):
        row = 0

    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']
    breed_name = dff.iloc[row]['breed']
    animal_name = dff.iloc[row]['name'] if 'name' in dff.columns and pd.notna(dff.iloc[row]['name']) else 'Unnamed Animal'

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed_name),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(animal_name),
                            html.P(f"Breed: {breed_name}")
                        ])
                    ]
                )
            ]
        )
    ]

# Run Dashboard Server
app.run_server(mode='external')

Successfully connected to the database.
Dash app running on https://chancerelax-antennapostage-3000.codio.io/proxy/8050/
